# Phase 5 - FSDP Scaling

**RHOAIENG-89619** (epic RHOAIENG-80952)

## What this notebook does

Run the same LoRA fine-tune of Llama-3.1-8B on Alpaca at 1, 2, 4, and 8 GPUs
as Phase 1, with one change: the model is sharded across GPUs with **FSDP**
instead of replicated with DDP. Everything else, the LoRA config, the batch
geometry, the step count and the metrics, is identical, so the two scaling
curves can be compared directly.

## The question

Phase 1 found near-linear scaling under DDP: 7.76x at 8 GPUs, 97.1% efficiency.
That holds because LoRA leaves only ~0.5% of parameters trainable, so the
gradient AllReduce carries very little.

FSDP does not work that way. It shards the model itself, so every step
all-gathers each layer's weights before using them, whether those weights are
trainable or not. The communication no longer scales with the adapter size, it
scales with the model. The expectation is that scaling falls further from
linear, and this measures by how much.

## Why LoRA and not a full fine-tune

The natural pairing for FSDP is full-parameter fine-tuning, but that cannot
produce a 1-GPU point. Llama-3.1-8B in bf16 needs roughly 16 GB of weights,
16 GB of gradients and 64 GB of Adam state, about 96 GB before activations
against 80 GB on an A100, and at world size 1 FSDP shards nothing. Keeping LoRA
preserves the baseline that the whole speedup curve is normalised against, and
changes exactly one variable against Phase 1.

## What to expect at 1 GPU

Accelerate only selects FSDP once it sees a multi-process launch, so the 1-GPU
run falls back to plain single-GPU training. That is the same thing Phase 1's
1-GPU run did, which is what makes it a valid shared baseline. `train_func`
records `fsdp_enabled` in the summary so this is visible rather than assumed.

## Why we keep the batch size constant

When you go from 1 GPU to 4 GPUs, each GPU still processes the same micro-batch.
Without adjustment, the effective batch size would quadruple (4 GPUs x 4
samples = 16 instead of 4). A larger batch changes the training dynamics -
different loss curve, different convergence - making it impossible to compare
runs fairly.

To keep things apples-to-apples, we fix the **global batch size** at 128 and
adjust **gradient accumulation** inversely:

| GPUs | Micro-batch per GPU | Grad accum steps | Effective batch |
|------|---------------------|------------------|-----------------|
| 1    | 4                   | 32               | 128             |
| 2    | 4                   | 16               | 128             |
| 4    | 4                   | 8                | 128             |
| 8    | 4                   | 4                | 128             |

This means every run sees the same number of tokens per optimizer step. The
only thing that changes is wall-clock time per step - which is exactly what
we want to measure.

## Why we pin to one node

All runs stay on a single 8-GPU node connected by NVLink (600 GB/s). If we
spread across two nodes, the GPUs would synchronize over Ethernet (12.5 GB/s)
- roughly 50x slower. FSDP is far more sensitive to this than DDP was, since
it communicates every layer of every step rather than the adapter gradients
once per step.

### Prerequisites

- OpenShift AI workbench with cluster access
- Namespace `dhryshch-elastic-scaling` with PVC `elastic-scaling-shared` and Secret `hf-token`
- All training pods pinned to a single 8-GPU node (NVLink domain)
- Phase 1 results in `results/phase1/scaling.csv` to compare against

Training code lives in `phase5/train.py`; `TransformersTrainer` serializes it via
`inspect.getsource()` and runs it on the cluster pods.


## Install the Kubeflow SDK

In [ ]:
!pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2"
!pip install yamlmagic hatchling --index-url https://pypi.org/simple
!pip install --no-deps .. --index-url https://pypi.org/simple
%load_ext yamlmagic

## Training Configuration

All parameters for the experiment in one place. Infrastructure settings
(namespace, PVC, node) are used by the submission logic below. Training
settings are passed through to `train_func` on the cluster pods.

Every value below matches Phase 1 except the two FSDP settings. Changing any
of the others breaks the comparison.

Key choices:
- **global_batch_size = 128**: kept constant across all GPU counts (see intro)
- **max_steps = 200**: enough to get stable throughput measurements after warmup
- **seq_length = 1024**: with packing enabled, every sequence is exactly 1024
  tokens, so throughput numbers are precise (no padding waste)
- **warmup_steps_excluded = 20**: first 20 steps are excluded from throughput
  averages because GPU caches, JIT compilation, and data pipeline warmup make
  them unrepresentatively slow
- **fsdp_sharding = full_shard auto_wrap**: shards parameters, gradients and
  optimizer state, wrapping each transformer block as its own FSDP unit
- **fsdp_layer_cls = LlamaDecoderLayer**: the block to wrap, model-specific


In [ ]:
%%yaml parameters

# Infrastructure
namespace: dhryshch-elastic-scaling
pvc_name: elastic-scaling-shared
hf_secret_name: hf-token
target_node: oai-kft-ibm-jcsbk-gpu-2-8gmgw

# Model & Data
model_id: meta-llama/Llama-3.1-8B
dataset_id: tatsu-lab/alpaca
output_dir: /mnt/kubeflow-checkpoints

# Training
max_steps: 200
save_steps: 100
seq_length: 1024
seed: 42
global_batch_size: 128
per_device_batch_size: 4
lora_r: 16
lora_alpha: 32
warmup_steps_excluded: 20

# FSDP
fsdp_sharding: full_shard auto_wrap
fsdp_layer_cls: LlamaDecoderLayer

# Sweep
gpu_counts: [1, 2, 4, 8]
repeats: 1

## Training Function

`TransformersTrainer` serializes `train_func` via `inspect.getsource()` - all
imports and logic must be inside the function body. This is how the SDK ships
your training code to the cluster pods without needing a custom container image.

The function lives in `train.py` for testability; `autoreload` picks up edits
without restarting the kernel.

> **Note:** `train.py` contains the same logic for standalone CLI usage.

In [ ]:
%load_ext autoreload
%autoreload 2

from elastic_scaling_poc.phase5.train import train_func

print("train_func loaded - self-contained for inspect.getsource()")

## Training Client

Authenticate to the cluster API. In an OpenShift AI workbench,
`NOTEBOOK_USER_TOKEN` is set automatically; otherwise falls back to the pod
service-account token.

TLS verification is disabled because the IBM cluster API uses a self-signed
certificate. In production, you would supply the CA bundle instead:
```python
config.ssl_ca_cert = "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt"
```

In [ ]:
import os
from pathlib import Path

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubernetes import client as k8s

api_server = os.environ["OPENSHIFT_API_URL"]
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
if not token:
    sa_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_path.exists():
        token = sa_path.read_text().strip()
if not token:
    raise RuntimeError(
        "Set NOTEBOOK_USER_TOKEN, or run inside a workbench "
        "with a service-account token."
    )

config = k8s.Configuration()
config.host = api_server
config.api_key = {"authorization": f"Bearer {token}"}
config.verify_ssl = False

client = TrainerClient(
    KubernetesBackendConfig(
        namespace=parameters["namespace"],
        client_configuration=config,
    )
)

## Training Job

Submit a `TransformersTrainer` for a given GPU count. Pod overrides:

- **Node selector** - pins all workers to the target node so GPUs communicate
  over NVLink, not Ethernet
- **PVC mount** - shared storage for HF model cache, checkpoints, and metrics
- **HF token** - injected from a Kubernetes Secret so the pod can download
  gated models (Llama 3.1) from Hugging Face
- **/dev/shm** - 16Gi memory-backed tmpfs required by NCCL for shared-memory
  inter-GPU communication (default 64MB is too small, and FSDP all-gathers every layer of every step)

### Resuming must be turned off explicitly

`TransformersTrainer` patches `Trainer.__init__` and overwrites the training
script's `output_dir` and `save_strategy` with its own. Left at its defaults it
points every run at the same directory (the root of the PVC mount) and then
auto-resumes from whatever checkpoint it finds there. In a sweep that means the
2, 4, and 8 GPU runs resume from the 1 GPU run's checkpoint near the final step,
train a single step, and stop - which is what invalidated the first results.

The fix has two parts. Each run writes to its own subdirectory in the `pvc://`
URL (`pvc://<pvc>/phase5-fsdp-<n>gpu/checkpoints`), so the GPU counts can no
longer see each other's checkpoints. And `train.py` passes
`resume_from_checkpoint=False`, which is what actually disables the SDK's
auto-resume - it only triggers on `None`.

That second part is load-bearing rather than belt-and-braces: repeats reuse the
same per-run directory, so without it a second repeat would resume from the
first one's checkpoint and reproduce the original bug.

In [ ]:
import time

from kubeflow.trainer.options import (
    ContainerOverride,
    Name,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig
from kubeflow_trainer_api.models import IoK8sApimachineryPkgApiResourceQuantity

PACKAGES = ["datasets", "peft", "trl", "nvidia-ml-py"]


def pod_overrides():
    return PodTemplateOverrides(
        PodTemplateOverride(
            target_jobs=["node"],
            spec=PodSpecOverride(
                node_selector={"kubernetes.io/hostname": parameters["target_node"]},
                volumes=[
                    {
                        "name": "dshm",
                        "emptyDir": {
                            "medium": "Memory",
                            "sizeLimit": IoK8sApimachineryPkgApiResourceQuantity("16Gi"),
                        },
                    },
                    {
                        "name": "hf-token",
                        "secret": {"secretName": parameters["hf_secret_name"]},
                    },
                ],
                containers=[
                    ContainerOverride(
                        name="node",
                        volume_mounts=[
                            {"name": "dshm", "mountPath": "/dev/shm"},
                            {"name": "hf-token", "mountPath": "/mnt/hf-token", "readOnly": True},
                        ],
                    ),
                ],
            ),
        )
    )


def submit_job(gpus, name=None):
    run_name = f"phase5-fsdp-{gpus}gpu"
    trainer = TransformersTrainer(
        func=train_func,
        func_args=parameters,
        num_nodes=1,
        resources_per_node={"nvidia.com/gpu": gpus},
        packages_to_install=PACKAGES,
        output_dir=f"pvc://{parameters['pvc_name']}/{run_name}/checkpoints",
        periodic_checkpoint_config=PeriodicCheckpointConfig(
            save_strategy="steps",
            save_steps=parameters["save_steps"],
            save_total_limit=1,
        ),
    )
    trainer.enable_jit_checkpoint = False
    runtime = client.backend.get_runtime("torch-distributed")
    options = [pod_overrides()]
    if name:
        options.append(Name(name))
    job_name = client.train(trainer=trainer, runtime=runtime, options=options)
    print(f"Submitted {job_name} ({gpus} GPU)")
    return job_name


def watch_job(name, poll_s=15):
    """Block until the job finishes. Interrupt the kernel to stop waiting."""
    seen, start = None, time.time()
    while True:
        status = client.get_job(name).status
        if status != seen:
            print(f"  [{int(time.time() - start):5d}s] {status}")
            seen = status
        if status in ("Complete", "Failed"):
            return status
        time.sleep(poll_s)

## Smoke Test - 2 GPUs

Two GPUs, not one: this is the cheapest run that actually exercises FSDP, and
the 1-GPU path never does. Check the log for `fsdp_enabled=True` before
starting the sweep. If it reads `False`, the sharding config is not taking
effect and the sweep would measure plain DDP.

Also verifies the runtime image has the right libraries, the PVC mounts work,
and the HF token reaches the pod.


In [ ]:
smoke = submit_job(2, name="phase5-smoke")

In [ ]:
job = client.get_job(name="phase5-smoke")
print(f"Job: {job.name}")
print(f"Status: {job.status}")

In [ ]:
for line in client.get_job_logs("phase5-smoke", follow=True):
    print(line, end="")

## The Sweep

Sequential on purpose - the runs compete for the same GPUs, and two runs
sharing a node would not be measuring what we think.

The 1-GPU run is the long pole: same work, one eighth the hardware.

Set `repeats` to 3 in the YAML config once timings look stable. Multiple
repeats let you report confidence intervals ("4 GPUs gives 3.7x +/- 0.1x
speedup") rather than a single lucky or unlucky sample.


In [ ]:
gpu_counts = parameters["gpu_counts"]
repeats = parameters["repeats"]

runs = []
for rep in range(repeats):
    for n in gpu_counts:
        name = f"phase5-{n}gpu-r{rep}"
        job = submit_job(n, name=name)
        status = watch_job(job)
        runs.append({"gpus": n, "repeat": rep, "job": job, "status": status})
        print(f"  -> {n} GPU rep {rep}: {status}")

runs

## Results

Each run writes `summary.json` to the PVC with per-step timing data.

The key metric is **scaling efficiency** - what percentage of the ideal
linear speedup you actually get:

```
efficiency = (actual_speedup / num_gpus) x 100%
```

Phase 1 reached 97.1% at 8 GPUs under DDP. The gap between that and what FSDP
reaches here is the cost of sharding the model, and is the answer this phase
was run for.

Three columns say where the cost lands. Rising `mean_step_time_s` with
`gpu_util_mean_pct` falling means time spent blocked on communication. Step
time rising while utilisation stays near 99% means the collectives are being
counted as busy work. `gpu_mem_peak_gb` should drop below Phase 1's flat
20.8 GB as GPUs are added, since that is what sharding buys.


In [ ]:
import json
import time

import pandas as pd

WORKBENCH_MOUNT = Path("/opt/app-root/src/elastic-scaling-shared")


def read_summaries():
    """Read this phase's summaries only, newest write time shown per run."""
    paths = sorted(WORKBENCH_MOUNT.glob("phase5-fsdp-*gpu/summary.json"))
    for p in paths:
        age_min = (time.time() - p.stat().st_mtime) / 60
        print(f"  {p.parent.name}: written {age_min:.0f} min ago")
    return [json.loads(p.read_text()) for p in paths]


summaries = read_summaries()
len(summaries)

In [ ]:
df = pd.DataFrame(summaries).sort_values("world_size").reset_index(drop=True)

single = df.loc[df.world_size == 1, "mean_tokens_per_s"]
if len(single):
    df["speedup"] = df.mean_tokens_per_s / single.iloc[0]
    df["efficiency_pct"] = (100 * df.speedup / df.world_size).round(1)

cols = [
    "world_size", "steady_state_steps", "mean_step_time_s", "mean_tokens_per_s",
    "gpu_util_mean_pct", "gpu_mem_peak_gb",
]
df[cols + [c for c in ("speedup", "efficiency_pct") if c in df]]

## Save & Cleanup

Results are a deliverable - save to the repo before cleaning up cluster resources.
The model, dataset, and checkpoints remain on the PVC.

In [ ]:
out = Path("../results/phase5")
out.mkdir(parents=True, exist_ok=True)
(out / "summaries.json").write_text(json.dumps(summaries, indent=2))
df.to_csv(out / "scaling.csv", index=False)
print("Wrote", out.resolve())

In [ ]:
for r in runs:
    try:
        client.delete_job(name=r["job"])
        print(f"Deleted {r['job']}")
    except Exception as e:
        print(f"Skip {r['job']}: {e}")